# Un singur model pentru ciorchini + trunchiuri (YOLO26m)

**Scop:** robotul să găsească mai mulți ciorchini și trunchiuri, și în vii pe care nu le-a văzut.

**Cum:** antrenăm un model cu 2 clase (`ciorchine`, `trunchi`) pe **8 dataseturi** din 6 țări:
label map (robot, Portugalia), WGISD (Brazilia), Pinheiro (Portugalia), wGrapeUNIPD (Italia), CERTH (Grecia),
HUMAIN-Lab (Grecia), EscaYard B7 (Spania), mană (frunziș fără trunchi).
Fiecare dataset are etichete doar pentru o clasă; clasa lipsă o completează modelele actuale, doar unde sunt foarte sigure (≥ 60%).

**Test cinstit** (poze nefolosite la antrenare): label map test (robot), WGISD test, Pinheiro test, Vairão (vie nouă),
EscaYard B9 (vie nouă). La final: tabel **vechi vs nou** și cel mai bun model salvat în Drive.

## Ce faceți voi (totul la început)
1. **Runtime → Change runtime type → T4 GPU → Save**, apoi **Runtime → Run all**.
2. Celula 2: **Connect to Google Drive**.
3. Celula 3: dacă modelele actuale nu sunt în Drive (`MyDrive/vie_hackathon/`), alegeți-le de pe laptop din `models/`:
   `model_ciorchine_v2.pt` și `model_trunchi_v2.pt`.
4. Celula 4: lipiți cheia API Roboflow și apăsați Enter.

**Durată:** descărcări ~15–25 min, set de date ~20 min, antrenare ~2–3 ore (progres pe epoci, cu procente), alegere ~20 min.
Totul se salvează în `MyDrive/vie_hackathon/bt_runs/`. Dacă sesiunea se închide, vedeți celula „Continuare”.

## 1. Placa video

In [ ]:
!nvidia-smi -L

## 2. Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/vie_hackathon"
os.makedirs(OUT, exist_ok=True)

## 3. Modelele actuale (profesori pentru clasa lipsă + comparație)

In [ ]:
from google.colab import files

for name in ("model_ciorchine_v2.pt", "model_trunchi_v2.pt"):
    while not os.path.exists(f"{OUT}/{name}"):
        print(f"{name} is not in Drive -> choose it from the laptop (folder models/)")
        for up, data in files.upload().items():
            for want in ("model_ciorchine_v2", "model_trunchi_v2"):
                if want in up:
                    open(f"{OUT}/{want}.pt", "wb").write(data)
    print("OK:", name, round(os.path.getsize(f"{OUT}/{name}") / 1e6), "MB")

## 4. label map v8 (Roboflow) — lipiți cheia API

In [ ]:
import json, urllib.request, zipfile
from getpass import getpass

key = getpass("Roboflow API key: ")
info = json.load(urllib.request.urlopen(f"https://api.roboflow.com/grape-detection/label-map/8/yolov8?api_key={key}"))
urllib.request.urlretrieve(info["export"]["link"], "/content/labelmap.zip")
zipfile.ZipFile("/content/labelmap.zip").extractall("/content/tv/data/labelmap")
print({s: len(os.listdir(f"/content/tv/data/labelmap/{s}/images")) for s in ("train", "valid", "test")})

## 5. Scripturile (aceleași ca în proiect, folderul `tools/`)

In [ ]:
!mkdir -p /content/bt

In [ ]:
%%writefile /content/trunk_adapt.py
"""Trunk model for new vineyards (model_trunchi_v2): multi-vineyard training + a "post" class + self-training on the
new vineyard's own unlabeled photos. Round 1 (robot view, Aveleda) keeps using the original model_trunchi.pt.

Data (classes: 0 = trunchi, 1 = stalp):
  - label map v8 (Aveleda, Portugal, robot camera)  data/labelmap/{train,valid,test}   trunk + truk -> trunchi
  - HUMAIN-Lab vine-trunk (3 vineyards, Greece)      data/humain/git/...                only the 629 original photos
  - posts: auto-labeled by YOLOE ("wooden post", "metal pole"), dropped where they overlap a labeled trunk
  - EscaYard block B7 (Spain): photos without labels -> trunks auto-labeled by the previous model (self-training)
Test: EscaYard block B9 is never used for training (different vines than B7).

Steps (from the project folder):
  python -m tools.trunk_adapt build                  # label map + HUMAIN + posts -> data/trunk_v2
  python -m tools.trunk_adapt train --name r0        # fine-tune from models/model_trunchi.pt
  python -m tools.trunk_adapt pseudo --weights runs/trunk/r0/weights/best.pt   # auto-label B7, add to the dataset
  python -m tools.trunk_adapt train --name r1 --init runs/trunk/r0/weights/best.pt
  python -m tools.trunk_adapt eval --weights runs/trunk/r1/weights/best.pt     # B9: photos with a trunk + drawn photos
"""
import argparse
import csv
import os
import random
import shutil
from pathlib import Path

import cv2

# Self-contained so the same file runs in Colab: TRUNK_ROOT = folder with data/ and models/, TRUNK_RUNS = where runs go
ROOT = Path(os.environ.get("TRUNK_ROOT", Path(__file__).resolve().parent.parent))
RUNS = Path(os.environ.get("TRUNK_RUNS", ROOT / "runs" / "trunk"))
DS = ROOT / "data" / "trunk_v2"
LM, HUM = ROOT / "data" / "labelmap", ROOT / "data" / "humain" / "git"
ESCA = ROOT / "data" / "escayard"
NAMES = {0: "trunchi", 1: "stalp"}
POST_PROMPTS = ["wooden post", "metal pole", "grapevine trunk", "tree trunk"]
MAX_SIDE = 1280  # EscaYard phone photos are shrunk to this before training / testing


def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def inside(a, b):
    """Which share of the smaller box lies inside the other one."""
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    smaller = min((a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1]))
    return inter / smaller if smaller > 0 else 0.0


def read_yolo(path):
    boxes = []
    if path.exists():
        for line in path.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                boxes.append((int(p[0]), *map(float, p[1:])))
    return boxes


def to_xyxy(b, w, h):
    _, cx, cy, bw, bh = b
    return ((cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h)


def to_line(c, box, w, h):
    x1, y1, x2, y2 = box
    return f"{c} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}"


def escayard_blocks():
    blocks = {"B7": [], "B9": []}
    with open(ESCA / "datasheet.csv", encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            if r["Vineyard"].strip() in blocks and r["File"].strip():
                blocks[r["Vineyard"].strip()].append(r["File"].strip())
    photos = {p.name: p for p in (ESCA / "photos").rglob("*.jpg")}
    return {b: [photos[n] for n in sorted(names) if n in photos] for b, names in blocks.items()}


def shrink(img):
    h, w = img.shape[:2]
    s = MAX_SIDE / max(h, w)
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img


def is_trunk_shape(box, w, h):
    """Trunks stand in the lower part of the photo and are taller than wide; posts reach the top wires."""
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    return bh >= 0.8 * bw and y2 >= 0.55 * h and bh <= 0.75 * h


def is_post_shape(box, w, h):
    x1, y1, x2, y2 = box
    return (y2 - y1) >= 2.5 * (x2 - x1)


class PostLabeler:
    def __init__(self):
        from ultralytics import YOLOE
        local = ROOT / "models" / "yoloe-11l-seg.pt"
        self.m = YOLOE(str(local) if local.exists() else "yoloe-11l-seg.pt")  # Ultralytics downloads it if missing
        self.m.set_classes(POST_PROMPTS, self.m.get_text_pe(POST_PROMPTS))

    def posts(self, img, trunks, conf=0.3):
        """Post boxes (xyxy) in img that do not overlap any trunk box."""
        h, w = img.shape[:2]
        r = self.m.predict(img, imgsz=1024, conf=conf, verbose=False)[0]
        out = []
        for b, k in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist()):
            if POST_PROMPTS[int(k)] in ("wooden post", "metal pole") and is_post_shape(b, w, h):
                if all(iou(b, t) < 0.1 and inside(b, t) < 0.3 for t in trunks):
                    out.append(b)
        return out


def write_item(split, name, img, lines):
    for sub in ("images", "labels"):
        (DS / split / sub).mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(DS / split / "images" / f"{name}.jpg"), img)
    (DS / split / "labels" / f"{name}.txt").write_text("\n".join(lines))


def write_yaml():
    (DS / "data.yaml").write_text(
        f"path: {DS.as_posix()}\ntrain: train/images\nval: valid/images\ntest: test/images\n"
        "names:\n  0: trunchi\n  1: stalp\n", encoding="utf-8")


def build(args):
    if DS.exists():
        shutil.rmtree(DS)
    posts = PostLabeler()
    sources = [("lm", LM / s / "images", LM / s / "labels", s, {2: 0, 3: 0}) for s in ("train", "valid", "test")]
    sources += [("hu", HUM / "train" / "Original_photos", HUM / "train" / "Original_photos", "train", {0: 0}),
                ("hu", HUM / "valid", HUM / "valid", "valid", {0: 0}),
                ("hu", HUM / "test", HUM / "test", "test", {0: 0})]
    n_posts = 0
    for prefix, img_dir, lbl_dir, split, cmap in sources:
        files = sorted(img_dir.glob("*.jpg"))
        for i, p in enumerate(files, 1):
            img = cv2.imread(str(p))
            h, w = img.shape[:2]
            trunks = [to_xyxy(b, w, h) for b in read_yolo(lbl_dir / f"{p.stem}.txt") if b[0] in cmap]
            lines = [to_line(0, t, w, h) for t in trunks]
            if split != "test":  # the test sets keep only their real labels
                found = posts.posts(img, trunks)
                n_posts += len(found)
                lines += [to_line(1, b, w, h) for b in found]
            write_item(split, f"{prefix}_{p.stem}", img, lines)
            if i % 200 == 0 or i == len(files):
                print(f"{prefix} {split}: {i}/{len(files)} ({i / len(files):.0%})", flush=True)
    write_yaml()
    print("Posts auto-labeled:", n_posts)


def pseudo(args):
    from ultralytics import YOLO
    m = YOLO(args.weights)
    posts = PostLabeler()
    b7 = escayard_blocks()["B7"]
    random.Random(0).shuffle(b7)
    n_valid = len(b7) // 6
    kept = skipped = 0
    for split in ("train", "valid"):  # replace the previous round's pseudo-labels
        for old in (DS / split / "images").glob("es_*"):
            old.unlink()
            (DS / split / "labels" / f"{old.stem}.txt").unlink(missing_ok=True)
    for i, p in enumerate(b7):
        img = shrink(cv2.imread(str(p)))
        h, w = img.shape[:2]
        r = m.predict(img, imgsz=args.imgsz, conf=args.conf, verbose=False)[0]
        trunks = [b for b, k in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist())
                  if int(k) == 0 and is_trunk_shape(b, w, h)]
        if not trunks:
            # the trunk is almost always there: a photo without a pseudo-label would teach "trunk = background"
            skipped += 1
            continue
        found_posts = posts.posts(img, trunks)
        lines = [to_line(0, t, w, h) for t in trunks] + [to_line(1, b, w, h) for b in found_posts]
        kept += len(trunks)
        write_item("valid" if i < n_valid else "train", f"es_{p.stem}", img, lines)
    print(f"EscaYard B7: {len(b7)} photos, {kept} pseudo-labeled trunks (conf >= {args.conf}, imgsz {args.imgsz}), "
          f"{skipped} photos without a trunk left out")


def train(args):
    from ultralytics import YOLO
    m = YOLO(args.init)
    m.train(data=str(DS / "data.yaml"), imgsz=args.imgsz, epochs=args.epochs, patience=args.patience, batch=args.batch,
            workers=2, project=str(RUNS), name=args.name, exist_ok=True, plots=False,
            hsv_h=0.03, hsv_s=0.8, hsv_v=0.5, degrees=5, scale=0.6, amp=args.amp)
    best = RUNS / args.name / "weights" / "best.pt"
    res = YOLO(str(best)).val(data=str(DS / "data.yaml"), split="test", imgsz=args.imgsz, verbose=False, plots=False)
    print(f"Test (label map + HUMAIN) trunk mAP50: {res.box.ap50[0]:.3f}")


def evaluate(args):
    from ultralytics import YOLO
    m = YOLO(args.weights)
    w = Path(args.weights)
    out = RUNS / f"eval_B9_{w.parent.parent.name if w.parent.name == 'weights' else w.stem}"
    out.mkdir(parents=True, exist_ok=True)
    b9 = escayard_blocks()["B9"]
    found = 0
    for p in b9:
        img = shrink(cv2.imread(str(p)))
        r = m.predict(img, imgsz=args.imgsz, conf=args.conf, verbose=False)[0]
        trunks = [(b, c) for b, c, k in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist()) if int(k) == 0]
        found += bool(trunks)
        for b, c in trunks:
            x1, y1, x2, y2 = map(int, b)
            cv2.rectangle(img, (x1, y1), (x2, y2), (30, 140, 255), 4)
            cv2.putText(img, f"trunk {c:.2f}", (x1, max(30, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (30, 140, 255), 3)
        cv2.imwrite(str(out / p.name), img)
    print(f"EscaYard B9 (never trained on): photos with a trunk {found}/{len(b9)} ({found / len(b9):.0%}) -> {out}")


def main():
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest="cmd", required=True)
    sub.add_parser("build")
    t = sub.add_parser("train")
    t.add_argument("--name", required=True)
    t.add_argument("--init", default=str(ROOT / "models" / "model_trunchi.pt"))
    t.add_argument("--imgsz", type=int, default=640)
    t.add_argument("--epochs", type=int, default=30)
    t.add_argument("--patience", type=int, default=8)
    t.add_argument("--batch", type=int, default=8)
    t.add_argument("--amp", action="store_true", help="mixed precision (use on Colab T4; GTX 16xx cards misbehave with it)")
    for name in ("pseudo", "eval"):
        s = sub.add_parser(name)
        s.add_argument("--weights", required=True)
        s.add_argument("--imgsz", type=int, default=640)
        s.add_argument("--conf", type=float, default=0.5 if name == "pseudo" else 0.4)
    args = p.parse_args()
    {"build": build, "train": train, "pseudo": pseudo, "eval": evaluate}[args.cmd](args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/bt/certh_sub.py
"""Takes 300 random CERTH training photos straight out of the 19 GB images.zip (HTTP range reads) -> certh/{images,labels}."""
import json, random, os, cv2, numpy as np
from concurrent.futures import ThreadPoolExecutor
from remotezip import RemoteZip
os.chdir('/content/bt')
d = json.load(open('certh_ann/annotations/mimc_train_images.json'))
imgs = random.Random(0).sample(d['images'], 300)
anns = {}
for a in d['annotations']:
    anns.setdefault(a['image_id'], []).append(a['bbox'])
os.makedirs('certh/images', exist_ok=True)
os.makedirs('certh/labels', exist_ok=True)
def job(im):
    z = RemoteZip('https://zenodo.org/api/records/10777647/files/images.zip/content')
    a = cv2.imdecode(np.frombuffer(z.read('images/multiple-instance-multiple-class/train_set/' + im['file_name']), np.uint8), cv2.IMREAD_COLOR)
    H, W = a.shape[:2]
    s = 1280 / max(H, W)
    a = cv2.resize(a, (round(W * s), round(H * s)), interpolation=cv2.INTER_AREA)
    stem = 'certh_' + im['file_name'][:-4]
    cv2.imwrite(f'certh/images/{stem}.jpg', a, [cv2.IMWRITE_JPEG_QUALITY, 90])
    lines = [f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}" for x, y, w, h in anns.get(im['id'], [])]
    open(f'certh/labels/{stem}.txt', 'w').write('\n'.join(lines))
done = 0
with ThreadPoolExecutor(6) as ex:
    for _ in ex.map(job, imgs):
        done += 1
        if done % 25 == 0:
            print(f'CERTH {done}/300', flush=True)
print('CERTH_OK')

In [ ]:
%%writefile /content/bt/build_bt.py
"""Builds the combined 2-class dataset (0 = ciorchine / bunch, 1 = trunchi / trunk) for one bunch+trunk model.

Runs in Colab (paths below). Every source is labeled for only some classes; the missing class is filled in by the
current models (teachers) where they are very sure, so the new model is not taught that a real trunk/bunch is background.

Train sources                        bunch labels              trunk labels
  label map v8 (robot, Portugal)     real (tiny + medium)      real
  WGISD (Brazil)                     real                      teacher (trunk v2)
  Pinheiro (Porto) - train part      real                      teacher
  wGrapeUNIPD (Italy)                real                      teacher
  CERTH 300 photos (Greece)          real                      teacher
  HUMAIN-Lab (Greece)                teacher (bunch v2)        real
  EscaYard B7 (Spain)                teacher                   teacher (trunk v2, same as the trunk round)
  mildew even photos                 teacher                   none (negatives for trunk)
Held-out tests (never trained): label map test, WGISD test, Pinheiro test part, Vairão (all), EscaYard B9, mildew odd.
"""
import glob
import json
import os
import random
import shutil
import sys
from pathlib import Path

import cv2

BT = Path("/content/bt")
TV = Path("/content/tv/data")
OUT = Path("/content/bt/ds")
DRIVE = Path("/content/drive/MyDrive/vie_hackathon")
T_BUNCH, T_TRUNK = 0.6, 0.6  # teacher confidence for filling in a missing class
MAX_SIDE = 1280


def shrink(img):
    h, w = img.shape[:2]
    s = MAX_SIDE / max(h, w)
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img


def yolo_boxes(path, keep):
    """Reads a YOLO label file; keep maps source class -> new class. Polygons are turned into boxes."""
    out = []
    if not Path(path).exists():
        return out
    for line in Path(path).read_text().split("\n"):
        p = line.split()
        if len(p) < 5 or int(p[0]) not in keep:
            continue
        v = list(map(float, p[1:]))
        if len(v) == 4:
            cx, cy, w, h = v
        else:  # segmentation polygon
            xs, ys = v[0::2], v[1::2]
            cx, cy, w, h = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
        out.append((keep[int(p[0])], cx, cy, w, h))
    return out


class Teachers:
    def __init__(self):
        from ultralytics import YOLO
        self.bunch = YOLO(str(DRIVE / "model_ciorchine_v2.pt"))
        self.trunk = YOLO(str(DRIVE / "model_trunchi_v2.pt"))

    def _run(self, m, img, imgsz, conf, cls_name, new_cls):
        r = m.predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]
        h, w = img.shape[:2]
        return [(new_cls, (b[0] + b[2]) / 2 / w, (b[1] + b[3]) / 2 / h, (b[2] - b[0]) / w, (b[3] - b[1]) / h)
                for b, k in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist()) if r.names[int(k)] == cls_name]

    def bunches(self, img):
        return self._run(self.bunch, img, 960, T_BUNCH, "ciorchine", 0)

    def trunks(self, img):
        return self._run(self.trunk, img, 1280, T_TRUNK, "trunchi", 1)


def write(split, name, img, boxes, copies=1):
    for sub in ("images", "labels"):
        (OUT / split / sub).mkdir(parents=True, exist_ok=True)
    lines = "\n".join(f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in boxes)
    for k in range(copies):
        n = name if copies == 1 else f"{name}_{k}"
        cv2.imwrite(str(OUT / split / "images" / f"{n}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        (OUT / split / "labels" / f"{n}.txt").write_text(lines)


def main():
    if OUT.exists():
        shutil.rmtree(OUT)
    T = Teachers()
    stats = {}

    def add(split, prefix, files, real, fill_bunch, fill_trunk, copies=1):
        n = 0
        files = list(files)
        for k, (img_path, lbl_path) in enumerate(files, 1):
            if k % 200 == 0:
                print(f"  {prefix} {split}: {k}/{len(files)} ({k / len(files):.0%})", flush=True)
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img = shrink(img)
            boxes = real(lbl_path) if real else []
            if fill_bunch:
                boxes += T.bunches(img)
            if fill_trunk:
                boxes += T.trunks(img)
            write(split, f"{prefix}_{Path(img_path).stem}", img, boxes, copies)
            n += 1
        stats[f"{prefix} {split}"] = n
        print(f"{prefix} {split}: {n}", flush=True)

    # label map v8: 0 medium, 1 tiny -> bunch ; 2 truk, 3 trunk -> trunk
    lm_keep = {0: 0, 1: 0, 2: 1, 3: 1}
    for split in ("train", "valid"):
        files = [(p, TV / "labelmap" / split / "labels" / f"{p.stem}.txt") for p in sorted((TV / "labelmap" / split / "images").glob("*.jpg"))]
        add(split, "lm", files, lambda l: yolo_boxes(l, lm_keep), False, False)

    # WGISD: train.txt (minus a validation part, same split as before) ; test.txt stays out
    wg = Path("/content/wgisd")
    ids = [l.strip() for l in open(wg / "train.txt") if l.strip()]
    random.Random(0).shuffle(ids)
    nv = round(len(ids) * 0.15)
    for split, part in (("train", ids[nv:]), ("valid", ids[:nv])):
        add(split, "wg", [(wg / "data" / f"{i}.jpg", wg / "data" / f"{i}.txt") for i in part],
            lambda l: yolo_boxes(l, {0: 0}), False, True)

    # Pinheiro, wGrapeUNIPD, CERTH: real bunch labels (paths found by glob, YOLO format)
    def yolo_pairs(root):
        """(photo, label) pairs: label in the sibling labels/ folder, else any .txt with the same name in the tree."""
        files = sorted(Path(root).rglob("*"))
        by_stem = {}
        for f in files:
            if f.suffix == ".txt" and f.stem != "classes":
                by_stem.setdefault(f.stem, f)
        pairs = []
        for img in files:
            if img.suffix.lower() in (".jpg", ".jpeg", ".png"):
                lbl = Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")
                if not lbl.exists():
                    lbl = img.with_suffix(".txt") if img.with_suffix(".txt").exists() else by_stem.get(img.stem)
                if lbl is not None and lbl.exists():
                    pairs.append((img, lbl))
        return pairs

    # Pinheiro: every photo exists 11 times (original + 10 augmented copies, e.g. "flip_IMG_...") and a few photos sit in
    # two splits. Keep only originals; any photo that appears in test/ is held out completely (bunch test set).
    import re
    pin_all = [(i, l) for i, l in yolo_pairs(BT / "pinheiro") if re.search(r"IMG_\d+_\d+", i.name)]
    base = lambda p: re.search(r"IMG_\d+_\d+", p.name).group()
    test_ids = {base(img) for img, _ in pin_all if "/test/" in str(img)}
    pin = [(img, lbl) for img, lbl in pin_all if img.name.startswith("IMG_") and base(img) not in test_ids]
    pin = list({base(img): (img, lbl) for img, lbl in pin}.values())  # one copy per photo
    random.Random(1).shuffle(pin)
    add("train", "pin", pin[: int(len(pin) * 0.85)], lambda l: yolo_boxes(l, {0: 0}), False, True)
    add("valid", "pin", pin[int(len(pin) * 0.85):], lambda l: yolo_boxes(l, {0: 0}), False, True)
    pin_test = list({base(img): (img, lbl) for img, lbl in pin_all if base(img) in test_ids and img.name.startswith("IMG_")}.values())
    tdir = BT / "test_pinheiro"
    for sub in ("images", "labels"):
        (tdir / sub).mkdir(parents=True, exist_ok=True)
    for img, lbl in pin_test:
        shutil.copy(img, tdir / "images" / img.name)
        shutil.copy(lbl, tdir / "labels" / lbl.name)
    print("pinheiro test (held out):", len(pin_test), flush=True)
    # wGrapeUNIPD has one class (white grape bunch), whatever its id
    add("train", "wgu", yolo_pairs(BT / "wgrape"), lambda l: yolo_boxes(l, {k: 0 for k in range(10)}), False, True)
    add("train", "certh", [(p, BT / "certh" / "labels" / f"{p.stem}.txt") for p in sorted((BT / "certh" / "images").glob("*.jpg"))],
        lambda l: yolo_boxes(l, {0: 0}), False, True)

    # HUMAIN (Greece): real trunks, teacher bunches
    hu = TV / "humain" / "git"
    for split, d in (("train", hu / "train" / "Original_photos"), ("valid", hu / "valid")):
        add(split, "hu", [(p, d / f"{p.stem}.txt") for p in sorted(d.glob("*.jpg"))], lambda l: yolo_boxes(l, {0: 1}), True, False)

    # EscaYard B7 (not B9!): both classes from teachers ; the 20 hand-labeled photos get their real trunks
    sys.path.insert(0, "/content")
    os.environ.setdefault("TRUNK_ROOT", "/content/tv")
    import trunk_adapt as t
    b7 = [str(p) for p in t.escayard_blocks()["B7"]]
    manual = json.load(open("/content/manual.json")) if Path("/content/manual.json").exists() else {}
    for p in b7:
        img = shrink(cv2.imread(p))
        boxes = T.bunches(img)
        if Path(p).name in manual:
            boxes += [(1, (a + c) / 2, (b + d) / 2, c - a, d - b) for a, b, c, d in manual[Path(p).name]]
            write("train", f"es_{Path(p).stem}", img, boxes, copies=2)
        else:
            boxes += T.trunks(img)
            write("train", f"es_{Path(p).stem}", img, boxes)
    print("es train:", len(b7), flush=True)

    # mildew even photos: canopy without trunks (bunch teacher only)
    mil = sorted(glob.glob("/content/tv/data/mildew/downy_mildew_images/*.jpg"))[0::2]
    add("train", "mil", [(Path(p), None) for p in mil], None, True, False)

    (OUT / "data.yaml").write_text(f"path: {OUT}\ntrain: train/images\nval: valid/images\nnames:\n  0: ciorchine\n  1: trunchi\n")
    n_tr = len(list((OUT / "train" / "images").glob("*")))
    n_va = len(list((OUT / "valid" / "images").glob("*")))
    print(f"DONE train {n_tr} valid {n_va}", json.dumps(stats))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/bt/eval_bt.py
"""Scores bunch/trunk models on held-out sets only (Colab paths). Usage:
  python eval_bt.py prepare                     # builds the test folders once
  python eval_bt.py new  <weights.pt>           # the new 2-class model (0 = ciorchine, 1 = trunchi)
  python eval_bt.py old                         # current models: model_ciorchine_v2 (bunch), model_trunchi_v2 (trunk)

Tests (never trained on): label map test (robot view: bunch + trunk), WGISD test (bunch), Pinheiro test (bunch),
Vairão (trunk, new vineyard), EscaYard B9 (trunk found per photo, bunch count vs people's count).
"""
import csv
import glob
import os
import shutil
import sys
from pathlib import Path

import cv2

BT, TV = Path("/content/bt"), Path("/content/tv/data")
TD = BT / "tests"
DRIVE = Path("/content/drive/MyDrive/vie_hackathon")
sys.path.insert(0, "/content")
os.environ.setdefault("TRUNK_ROOT", "/content/tv")


def remap(src_lbl, dst_lbl, keep):
    lines = []
    if Path(src_lbl).exists():
        for line in Path(src_lbl).read_text().split("\n"):
            p = line.split()
            if len(p) >= 5 and int(p[0]) in keep:
                v = list(map(float, p[1:]))
                if len(v) > 4:
                    xs, ys = v[0::2], v[1::2]
                    v = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)]
                lines.append(f"{keep[int(p[0])]} " + " ".join(f"{x:.6f}" for x in v))
    Path(dst_lbl).write_text("\n".join(lines))


def make(name, pairs, keep):
    d = TD / name
    for sub in ("images", "labels"):
        (d / sub).mkdir(parents=True, exist_ok=True)
    for i, (img, lbl) in enumerate(pairs):
        stem = f"{i:05d}_{Path(img).stem}"
        shutil.copy(img, d / "images" / f"{stem}{Path(img).suffix}")
        remap(lbl, d / "labels" / f"{stem}.txt", keep)
    (d / "data.yaml").write_text(f"path: {d}\ntrain: images\nval: images\nnames:\n  0: ciorchine\n  1: trunchi\n")
    print(name, len(pairs))


def prepare():
    shutil.rmtree(TD, ignore_errors=True)
    lm = TV / "labelmap" / "test"
    make("labelmap_robot", [(p, lm / "labels" / f"{p.stem}.txt") for p in sorted((lm / "images").glob("*.jpg"))], {0: 0, 1: 0, 2: 1, 3: 1})
    wg = Path("/content/wgisd")
    make("wgisd", [(wg / "data" / f"{i.strip()}.jpg", wg / "data" / f"{i.strip()}.txt") for i in open(wg / "test.txt") if i.strip()], {0: 0})
    pt = BT / "test_pinheiro"
    make("pinheiro", [(p, pt / "labels" / f"{p.stem}.txt") for p in sorted((pt / "images").glob("*.jpg"))], {0: 0})
    va = [Path(p) for p in glob.glob("/content/vairao/**/images/*.png", recursive=True)]
    make("vairao", [(p, p.parent.parent / "labels" / f"{p.stem}.txt") for p in sorted(va)], {0: 1})


TEST_IDS = {"ciorchine": 0, "trunchi": 1}  # class ids in the test labels


def ap50(weights, test, cls, imgsz):
    """AP50 of one class. Models number their classes differently (old trunk model: 0 = trunchi), so the test labels
    are rewritten into the model's own numbering, keeping only the boxes of that class."""
    from ultralytics import YOLO
    model = YOLO(weights)
    names = dict(model.names)
    idx = next(k for k, v in names.items() if v == cls)
    src, d = TD / test, TD / f"_{test}_{cls}_{idx}_{len(names)}"
    if not d.exists():
        (d / "labels").mkdir(parents=True)
        try:
            os.symlink(src / "images", d / "images")
        except OSError:  # Windows without symlink rights
            shutil.copytree(src / "images", d / "images")
        for lbl in (src / "labels").glob("*.txt"):
            keep = [f"{idx} " + " ".join(l.split()[1:]) for l in lbl.read_text().split("\n")
                    if l.split() and int(l.split()[0]) == TEST_IDS[cls]]
            (d / "labels" / lbl.name).write_text("\n".join(keep))
        (d / "data.yaml").write_text(f"path: {d}\ntrain: images\nval: images\nnames:\n"
                                     + "".join(f"  {k}: {v}\n" for k, v in sorted(names.items())))
    r = model.val(data=str(d / "data.yaml"), imgsz=imgsz, batch=8, verbose=False, plots=False)
    return float(r.box.ap50[list(r.box.ap_class_index).index(idx)]) if idx in list(r.box.ap_class_index) else 0.0


def escayard(bunch_w, trunk_w, bunch_cls, trunk_cls, imgsz_b=1280, imgsz_t=1280):
    import trunk_adapt as t
    from ultralytics import YOLO
    rows = {r["File"].strip(): r for r in csv.DictReader(open(t.ESCA / "datasheet.csv", encoding="utf-8-sig"))}
    b9 = t.escayard_blocks()["B9"]
    mb, mt = YOLO(bunch_w), YOLO(trunk_w)
    found, errs = 0, []
    for p in b9:
        img = t.shrink(cv2.imread(str(p)))
        rt = mt.predict(img, imgsz=imgsz_t, conf=0.3, verbose=False)[0]
        found += any(rt.names[int(k)] == trunk_cls for k in rt.boxes.cls.tolist())
        n_true = rows[p.name]["Number of grape clusters"].strip()
        if n_true:
            rb = mb.predict(img, imgsz=imgsz_b, conf=0.4, verbose=False)[0]
            n = sum(rb.names[int(k)] == bunch_cls for k in rb.boxes.cls.tolist())
            errs.append(abs(n - int(n_true)))
    return found, len(b9), (sum(errs) / len(errs) if errs else float("nan")), len(errs)


def report(bunch_w, trunk_w, bunch_cls="ciorchine", trunk_cls="trunchi", imgsz_b=960, imgsz_t=1280, imgsz_es=None):
    """imgsz_es: size for counting bunches on EscaYard (the old bunch model was measured at 1280 there)."""
    out = {
        "robot bunch AP50": ap50(bunch_w, "labelmap_robot", bunch_cls, imgsz_b),
        "robot trunk AP50": ap50(trunk_w, "labelmap_robot", trunk_cls, imgsz_t),
        "WGISD bunch AP50": ap50(bunch_w, "wgisd", bunch_cls, imgsz_b),
        "Pinheiro bunch AP50": ap50(bunch_w, "pinheiro", bunch_cls, imgsz_b),
        "Vairao trunk AP50": ap50(trunk_w, "vairao", trunk_cls, imgsz_t),
    }
    f, n, err, nc = escayard(bunch_w, trunk_w, bunch_cls, trunk_cls, imgsz_es or imgsz_b, imgsz_t)
    out["EscaYard B9 trunk found %"] = f / n if n else 0.0
    out[f"EscaYard B9 bunch count error ({nc} vines)"] = round(err, 2)
    for k, v in out.items():
        print(f"{k:40s} {v:.3f}" if isinstance(v, float) else f"{k:40s} {v}", flush=True)
    return out


def score(out):
    """One number to pick the best checkpoint: average of the AP50s and the share of B9 photos with a trunk."""
    return sum(v for k, v in out.items() if "AP50" in k or "found" in k) / 6


if __name__ == "__main__":
    cmd = sys.argv[1]
    if cmd == "prepare":
        prepare()
    elif cmd == "old":
        report(str(DRIVE / "model_ciorchine_v2.pt"), str(DRIVE / "model_trunchi_v2.pt"), imgsz_es=1280)
    elif cmd == "new":
        size = int(sys.argv[3]) if len(sys.argv) > 3 else 960
        report(sys.argv[2], sys.argv[2], imgsz_b=size, imgsz_t=size)

In [ ]:
%%writefile /content/bt/setup.sh
#!/bin/bash
# Downloads every dataset for the bunch+trunk model into a fresh Colab runtime, all in parallel.
# Each download writes /content/bt/dl_<name>.log, ending with "<NAME>_OK" on success and "END" when it stops.
# label map v8 is NOT here: it needs the Roboflow key, pasted by the user in a notebook cell (getpass).
mkdir -p /content/bt /content/tv/data/escayard/photos /content/tv/data/mildew
cd /content/bt

run() {  # run <name> <command...> in the background
  local name=$1; shift
  ( ( "$@" ) > "dl_$name.log" 2>&1; echo END >> "dl_$name.log" ) &
}

run pip bash -c 'pip -q install ultralytics==8.4.163 py7zr remotezip && echo PIP_OK'
run wgisd bash -c 'git clone -q --depth 1 https://github.com/thsant/wgisd.git /content/wgisd && echo WGISD_OK'
run pinheiro bash -c 'curl -sfL -o pinheiro.zip "https://zenodo.org/api/records/7717055/files/GrapevineBunchDetection.zip/content" \
  && unzip -q pinheiro.zip -d pinheiro && rm pinheiro.zip && echo PINHEIRO_OK'
run wgrape bash -c 'curl -sfL -o wgrape.zip "https://zenodo.org/api/records/4066730/files/wGrapeUNIPD-DL%20dataset.zip/content" \
  && unzip -q wgrape.zip -d wgrape && rm wgrape.zip && echo WGRAPE_OK'
run vairao bash -c 'curl -sfL -o /content/vairao.zip "https://zenodo.org/api/records/18152298/files/Dataset_Vairao_Grapevine_Trunks.zip/content" \
  && unzip -q /content/vairao.zip -d /content/vairao && rm /content/vairao.zip && echo VAIRAO_OK'
run humain bash -c 'curl -sfL -o humain.zip "https://github.com/humain-lab/vine-trunk/raw/master/Humain-Lab-vine-trunk-dataset.zip" \
  && unzip -q humain.zip -d /content/tv/data/humain && rm humain.zip && echo HUMAIN_OK'
run escayard bash -c 'curl -sfL -o /content/tv/data/escayard/datasheet.csv "https://zenodo.org/api/records/10362568/files/datasheet.csv/content" \
  && curl -sfL -o escayard.7z "https://zenodo.org/api/records/10362568/files/PLANTs_SmartPhonePhotos_JPG.7z/content" \
  && while ! grep -q PIP_OK dl_pip.log 2>/dev/null; do sleep 5; done \
  && python3 -c "import py7zr; py7zr.SevenZipFile(\"escayard.7z\").extractall(\"/content/tv/data/escayard/photos\")" \
  && rm escayard.7z && echo ESCAYARD_OK'
run mildew bash -c 'cd /content/tv/data/mildew && wget -q -U "Mozilla/5.0" -O m.zip \
  "https://data.mendeley.com/public-files/datasets/yxf2yv5ymt/files/baa9a436-bd95-425a-8d6f-aca7152cb1b0/file_downloaded" \
  && unzip -q m.zip && rm m.zip && echo MILDEW_OK'
run certh bash -c 'curl -sfL -o certh_ann.zip "https://zenodo.org/api/records/10777647/files/annotations.zip/content" \
  && unzip -q certh_ann.zip -d certh_ann \
  && while ! grep -q PIP_OK dl_pip.log 2>/dev/null; do sleep 5; done \
  && python3 /content/bt/certh_sub.py'
echo STARTED

In [ ]:
%%writefile /content/manual.json
{"IMG_9457.jpg":[[0.1654,0.8322,0.2527,0.994],[0.8653,0.7092,0.9169,0.8001]],"IMG_20220714_214202.jpg":[[0.276,0.7147,0.3849,1],[0.8096,0.4886,0.8712,0.6017]],"IMG_20220714_212249.jpg":[[0.1488,0.6726,0.2352,0.9652]],"IMG_9479.jpg":[[0.3175,0.6504,0.4771,0.963],[0.0989,0.5174,0.1721,0.6394]],"IMG_20220714_213134.jpg":[[0.064,0.5208,0.256,0.9264],[0.8146,0.5341,0.9027,0.707]],"IMG_20220714_214717.jpg":[[0.1147,0.6105,0.2685,0.9054]],"IMG_20220714_212835.jpg":[[0.1995,0.7302,0.2976,0.9818]],"IMG_9455.jpg":[[0.6309,0.1805,0.6941,0.3678],[0.4838,0.1384,0.5395,0.2503],[0.4347,0.1062,0.4755,0.1805]],"IMG_20220714_212515.jpg":[[0.2552,0.7347,0.335,0.9896]],"IMG_20220714_214637.jpg":[[0.1538,0.6937,0.2328,0.9896]],"IMG_20220714_212558.jpg":[[0.2028,0.6637,0.2993,0.9974]],"IMG_20220714_212141.jpg":[[0.1214,0.6538,0.1729,0.932]],"IMG_20220714_212533.jpg":[[0.724,0.3689,0.7573,0.462],[0.3948,0.7845,0.4298,0.9209]],"IMG_9431.jpg":[[0.1588,0.7103,0.2469,0.9453],[0.8462,0.5995,0.9219,0.738]],"IMG_20220714_214536.jpg":[[0.6509,0.5551,0.6974,0.6238],[0.3466,0.8001,0.3915,0.8898]],"IMG_20220714_214047.jpg":[[0.2436,0.6892,0.3067,0.9419],[0.1114,0.4532,0.1388,0.5119]],"IMG_20220714_213914.jpg":[[0.0923,0.3955,0.1197,0.4565],[0.724,0.4332,0.7556,0.5041]],"IMG_9427.jpg":[[0.6301,0.421,0.66,0.5518],[0.266,0.4908,0.2959,0.595],[0.3092,0.4698,0.3333,0.5163],[0.0773,0.4731,0.0948,0.5263]],"IMG_9483.jpg":[[0.2959,0.165,0.3732,0.2991],[0.3932,0.6072,0.4281,0.6881]],"IMG_20220714_213458.jpg":[[0.0765,0.4432,0.1147,0.533],[0.3333,0.6926,0.3533,0.8377]]}

## 6. Descărcarea dataseturilor (toate deodată, ~15–25 min)

In [ ]:
import glob, subprocess, time

subprocess.run(["bash", "/content/bt/setup.sh"], check=True)
names = ["pip", "wgisd", "pinheiro", "wgrape", "vairao", "humain", "escayard", "mildew", "certh"]
t0 = time.time()
while True:
    logs = {n: open(f"/content/bt/dl_{n}.log").read() if os.path.exists(f"/content/bt/dl_{n}.log") else "" for n in names}
    done = [n for n in names if "END" in logs[n]]
    print(f"{time.time() - t0:5.0f}s  finished {len(done)}/{len(names)} ({len(done) / len(names):.0%})  waiting: "
          + ", ".join(n for n in names if n not in done), flush=True)
    if len(done) == len(names):
        break
    time.sleep(30)
failed = [n for n in names if "_OK" not in logs[n]]
print("FAILED:", failed if failed else "none")
for n in failed:
    print(f"--- {n}\n" + logs[n][-800:])

## 7. Setul combinat (~20 min: modelele actuale completează clasa lipsă)

In [ ]:
!cd /content/bt && python build_bt.py

## 8. Testele cinstite + scorul modelelor actuale (~10 min)

In [ ]:
import sys
sys.path.insert(0, "/content/bt")
import eval_bt as E

E.prepare()
old = E.report(f"{OUT}/model_ciorchine_v2.pt", f"{OUT}/model_trunchi_v2.pt", imgsz_es=1280)
json.dump(old, open(f"{OUT}/bt_old_scores.json", "w"), indent=1)

## 9. Antrenare YOLO26m la 960 (~2–3 ore, progres pe epoci)

In [ ]:
import time
from ultralytics import YOLO

T0 = time.time()
RUN = f"{OUT}/bt_runs/bt_m960"

def print_epoch(trainer):
    """After every epoch: one clear line with progress, score and time left."""
    ep, total = trainer.epoch + 1, trainer.epochs
    if ep > total:  # the final validation after training calls this once more
        return
    elapsed = time.time() - (getattr(trainer, "train_time_start", None) or T0)
    left = elapsed / ep * (total - ep)
    map50 = trainer.metrics.get("metrics/mAP50(B)", 0.0)
    best = getattr(getattr(trainer, "stopper", None), "best_epoch", ep)  # Ultralytics already counts from 1
    print(f"==> Epoch {ep}/{total} ({ep / total:.0%}) | validation mAP50: {map50:.3f} | best: epoch {best} | "
          f"elapsed: {elapsed / 60:.1f} min | left: max ~{left / 60:.0f} min", flush=True)

model = YOLO("yolo26m.pt")  # pre-trained on COCO, only fine-tuning
model.add_callback("on_fit_epoch_end", print_epoch)
model.train(data="/content/bt/ds/data.yaml", imgsz=960, epochs=30, patience=10, batch=8, workers=4,
            save_period=2, plots=False, project=f"{OUT}/bt_runs", name="bt_m960", exist_ok=True)

### Continuare (doar dacă sesiunea s-a închis în timpul antrenării)
Rulați din nou celulele 1–8 (ca să existe datele), apoi scoateți `#` din linia de mai jos: antrenarea continuă de unde a rămas.

In [ ]:
# m = YOLO(f"{OUT}/bt_runs/bt_m960/weights/last.pt"); m.add_callback("on_fit_epoch_end", print_epoch); m.train(resume=True)

## 10. Alegem cea mai bună epocă pe testele cinstite (~20 min)
Scor = media dintre AP50 (robot ciorchini, robot trunchiuri, WGISD, Pinheiro, Vairão) și procentul de poze B9 cu trunchi găsit.
Cel mai bun model se salvează ca `MyDrive/vie_hackathon/model_bunch_trunk.pt`.

In [ ]:
import shutil

W = f"{OUT}/bt_runs/bt_m960/weights"
epochs = sorted(glob.glob(f"{W}/epoch*.pt"), key=lambda p: int(p.split("epoch")[-1][:-3]))
candidates = [f"{W}/best.pt", f"{W}/last.pt"] + epochs[len(epochs) // 2:]
results = {}
for i, c in enumerate(candidates, 1):
    if os.path.exists(c):
        print(f"\n=== {i}/{len(candidates)} ({i / len(candidates):.0%}): {os.path.basename(c)}", flush=True)
        results[os.path.basename(c)] = E.report(c, c, imgsz_b=960, imgsz_t=960)
best = max(results, key=lambda k: E.score(results[k]))

keys = list(old)
print(f"\n{'':42s}{'old models':>12s}{'new ' + best:>18s}")
for k in keys:
    print(f"{k:42s}{old[k]:12.3f}{results[best][k]:18.3f}")
print(f"{'score (higher = better)':42s}{E.score(old):12.3f}{E.score(results[best]):18.3f}")
shutil.copy(f"{W}/{best}", f"{OUT}/model_bunch_trunk.pt")
json.dump(results, open(f"{OUT}/bt_new_scores.json", "w"), indent=1)
print("\nSaved:", f"{OUT}/model_bunch_trunk.pt", "(from", best + ")")